### Generative AI- Customer Churn Reports

This notebook implements two types of templates using Gemini API to generate customer churn risk summary and retention recommendation, The generated reports will be tested and evaluated to identify which prompt template produces more suitable output.

**Prompt Templates:

-- T1- Zero-Shot: Generates repots using instructions without examples

-- T2- Few-Shot: Generates repots using instructions with worked examples

Both templates use the same instructions and customer information to support a fair comparision

## Generative AI Model Selection & Setup

### Model Selection and Rationale

Google's Gemini 3.5 Flash-Lite is a multimodal large language model (LLM) developed by Google. It was selected as the generative AI model for the customer churn prediction and retention system.

The model was selected based on its low latency, cost efficiency, and suitability for repeated data processing tasks, which align with the project's requirements for generating concise churn risk reports and personalized retention recommendations.

In this project, Gemini interprets churn predictions, churn probabilities, individual customer features, K-Means behavioral segments, GMM purchase tiers, and GMM membership confidence to generate meaningful business insights and actionable retention recommendations.

Additionally, using the same generative AI model for both Zero-Shot and Few-Shot prompting supports a fair comparison between the two prompting techniques.

### Model Setup and Integration

The Gemini model is integrated into the project using the Google Gen AI SDK (`google-genai`) in a Python environment within a Jupyter Notebook.

The Gemini API key is stored in a `.env` file instead of being hardcoded in the source code for security purposes. The `python-dotenv` library is used to load the environment variables and retrieve the API key.

The API configuration and client initialization are implemented in a separate Python module (`api_config_template.py`) to improve code organization, maintainability, and reusability. The selected model is specified through the `MODEL_NAME` variable.

The prompt templates are populated with customer information and machine learning outputs before being submitted to the model using the `generate_content()` method. The generated text is then retrieved for evaluation and comparison between the two prompting techniques.

In [2]:
from pathlib import Path
from string import Formatter

# Load prompt templates
zero_shot_template = Path(
    "Generative_AI/prompts/zero_shot_template.txt"
).read_text(encoding="utf-8")

few_shot_template = Path(
    "Generative_AI/prompts/few_shot_template.txt"
).read_text(encoding="utf-8")

print("Both prompt templates loaded successfully.")

# Inspect template placeholders
for name, template in [
    ("Zero-Shot", zero_shot_template),
    ("Few-Shot", few_shot_template)
]:
    fields = {
        field for _, field, _, _ in Formatter().parse(template)
        if field is not None
    }
    print(f"\n{name} placeholders:")
    print(sorted(fields))

Both prompt templates loaded successfully.

Zero-Shot placeholders:
['cart_abandonment_rate', 'churn_prediction', 'churn_probability', 'gmm_confidence', 'gmm_persona', 'kmeans_persona', 'loyalty_score', 'recency_days']

Few-Shot placeholders:
['cart_abandonment_rate', 'churn_prediction', 'churn_probability', 'gmm_confidence', 'gmm_persona', 'kmeans_persona', 'loyalty_score', 'recency_days']


## Implementation & API Integration Code

In [3]:
from Generative_AI.api_config_template import client, MODEL_NAME

#Prepare function to generate content using the Gemini API
def generate_customer_reports(customer_data):

    # Fill the prompts with the customer data
    filled_zero_shot = zero_shot_template.format(**customer_data)
    filled_few_shot = few_shot_template.format(**customer_data)

    # Generate zero-shot report
    zero_shot_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=filled_zero_shot
    )
    zero_shot_report = zero_shot_response.text

    # Generate few-shot report
    few_shot_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=filled_few_shot
    )
    few_shot_report = few_shot_response.text

    # Return both reports
    return {
        "zero_shot_report": zero_shot_report,
        "few_shot_report": few_shot_report
    }

Gemini API key loaded successfully!


## Testing Framework & Output Comparison

### Loading the Supervised Learning Model

In [6]:
import joblib

lr_model = joblib.load(
    "Supervised_Learning/models/logistic_regression.joblib"
)

print(lr_model)
print(lr_model.classes_)
print(lr_model.n_features_in_)

LogisticRegression(max_iter=1000)
[0 1]
30


### Loading Test Data

In [10]:
import pandas as pd

test_original = pd.read_csv("Dataset/Processed/test_selected.csv")
test_processed = pd.read_csv("Dataset/Processed/test_processed.csv")

X_test = test_processed.drop(columns=["churn_flag"])

print("Original test data:", test_original.shape)
print("Processed test data:", test_processed.shape)
print("Model input:", X_test.shape)

print("Labels match:",
      test_original["churn_flag"].equals(test_processed["churn_flag"]))

Original test data: (20000, 16)
Processed test data: (20000, 31)
Model input: (20000, 30)
Labels match: True


### Data Alignment Verification

In [11]:
print("Features match:",
      X_test.columns.tolist() == lr_model.feature_names_in_.tolist())

Features match: True


In [12]:
for col in ["age_group", "gender", "region",
            "customer_segment", "preferred_channel"]:

    encoded = pd.get_dummies(
        test_original[col],
        prefix=f"categorical__{col}",
        dtype=int
    )

    match = (
        encoded == X_test[encoded.columns]
    ).all().all()

    print(col, ":", match)

age_group : True
gender : True
region : True
customer_segment : True
preferred_channel : True


In [13]:
import numpy as np
from sklearn.preprocessing import StandardScaler

train_original = pd.read_csv(
    "Dataset/Processed/train_selected.csv"
)

numerical_cols = [
    col.removeprefix("numerical__")
    for col in X_test.columns
    if col.startswith("numerical__")
]

scaler = StandardScaler()
scaler.fit(train_original[numerical_cols])

expected = scaler.transform(test_original[numerical_cols])

actual = X_test[
    ["numerical__" + col for col in numerical_cols]
].to_numpy()

print("Numerical features match:", np.allclose(expected, actual))

Numerical features match: True


### Generating Churn Predictions

In [14]:
churn_predictions = lr_model.predict(X_test)
churn_probabilities = lr_model.predict_proba(X_test)[:, 1]

print("Predictions:", churn_predictions[:5])
print("Probabilities:", churn_probabilities[:5])

Predictions: [1 0 0 0 0]
Probabilities: [6.16802637e-01 2.41536965e-01 6.27144201e-05 6.23019969e-05
 7.84967104e-08]


### Dataset-Based Prompt Testing

#### Loading Customer and Clustering Data

In [15]:
original_data = pd.read_csv(
    "Dataset/retail_customer_churn.csv"
)

clustered_data = pd.read_csv(
    "Dataset/clustered_data.csv"
)

print("Original dataset:", original_data.shape)
print("Clustered dataset:", clustered_data.shape)

print("\nOriginal dataset columns:")
print(original_data.columns.tolist())

print("\nClustered dataset columns:")
print(clustered_data.columns.tolist())

Original dataset: (100000, 18)
Clustered dataset: (100000, 23)

Original dataset columns:
['customer_id', 'age_group', 'gender', 'region', 'customer_segment', 'preferred_channel', 'purchase_frequency', 'avg_order_value', 'total_spent', 'recency_days', 'website_visits', 'discount_usage_rate', 'email_open_rate', 'cart_abandonment_rate', 'loyalty_score', 'engagement_score', 'churn_risk', 'churn_flag']

Clustered dataset columns:
['customer_id', 'age_group', 'gender', 'region', 'customer_segment', 'preferred_channel', 'purchase_frequency', 'avg_order_value', 'total_spent', 'recency_days', 'website_visits', 'discount_usage_rate', 'email_open_rate', 'cart_abandonment_rate', 'loyalty_score', 'engagement_score', 'churn_risk', 'churn_flag', 'kmeans_cluster', 'kmeans_segment', 'gmm_cluster', 'gmm_confidence', 'gmm_persona']


### Recovering Customer IDs

In [16]:
from sklearn.model_selection import train_test_split

_, test_indices = train_test_split(
    original_data.index,
    test_size=0.20,
    random_state=42,
    stratify=original_data["churn_flag"]
)

reconstructed_test = original_data.loc[
    test_indices
].reset_index(drop=True)

# Verify alignment with the saved test data
alignment_match = np.allclose(
    reconstructed_test[test_original.select_dtypes(include="number").columns],
    test_original.select_dtypes(include="number")
) and reconstructed_test[
    test_original.select_dtypes(exclude="number").columns
].equals(
    test_original.select_dtypes(exclude="number")
)

print("Test data alignment:", alignment_match)
print("Unique customer IDs:",
      reconstructed_test["customer_id"].nunique())

Test data alignment: True
Unique customer IDs: 20000


### Integrating Supervised and Unsupervised Learning Outputs

In [17]:
# Combine customer features and supervised learning outputs
test_data = reconstructed_test[
    ["customer_id", "recency_days", "loyalty_score",
     "cart_abandonment_rate"]
].copy()

test_data["churn_prediction"] = churn_predictions
test_data["churn_probability"] = churn_probabilities

# Integrate clustering outputs using customer ID
test_data = test_data.merge(
    clustered_data[
        ["customer_id", "kmeans_segment",
         "gmm_persona", "gmm_confidence"]
    ],
    on="customer_id",
    how="left",
    validate="one_to_one"
)

test_data = test_data.rename(
    columns={"kmeans_segment": "kmeans_persona"}
)

print("Integrated test data:", test_data.shape)

print("\nMissing values:")
print(test_data.isnull().sum())

display(test_data.head())

Integrated test data: (20000, 9)

Missing values:
customer_id              0
recency_days             0
loyalty_score            0
cart_abandonment_rate    0
churn_prediction         0
churn_probability        0
kmeans_persona           0
gmm_persona              0
gmm_confidence           0
dtype: int64


,customer_id,recency_days,loyalty_score,cart_abandonment_rate,churn_prediction,churn_probability,kmeans_persona,gmm_persona,gmm_confidence
0,C025120,131,14,0.57,1,6.168026e-01,High-Volume Buyers,Regular Buyers,0.997254
1,C084953,242,80,0.36,0,2.415370e-01,High-Volume Buyers,Heavy Users,1.000000
2,C037030,60,22,0.25,0,6.271442e-05,High-Volume Buyers,Heavy Users,0.999960
3,C032014,109,40,0.11,0,6.230200e-05,High-Volume Buyers,Heavy Users,0.999953
4,C086567,18,59,0.17,0,7.849671e-08,Engaged Light Buyers,Regular Buyers,0.996285


### Test Case Selection

In [18]:
print("Churn probability distribution:")
print(test_data["churn_probability"].describe())

print("\nK-Means persona distribution:")
print(test_data["kmeans_persona"].value_counts())

print("\nGMM persona distribution:")
print(test_data["gmm_persona"].value_counts())

print("\nGMM confidence distribution:")
print(test_data["gmm_confidence"].describe())

print("\nCustomers with GMM confidence below 80%:")
print((test_data["gmm_confidence"] < 0.80).sum())

Churn probability distribution:
count    2.000000e+04
mean     4.610880e-01
std      4.588254e-01
min      1.197998e-10
25%      4.941424e-04
50%      2.595926e-01
75%      9.962822e-01
max      1.000000e+00
Name: churn_probability, dtype: float64

K-Means persona distribution:
kmeans_persona
High-Volume Buyers          7534
Low-Engagement Customers    6307
Engaged Light Buyers        6159
Name: count, dtype: int64

GMM persona distribution:
gmm_persona
Regular Buyers       7568
Heavy Users          4996
Occasional Buyers    4436
One-time Buyers      3000
Name: count, dtype: int64

GMM confidence distribution:
count    20000.000000
mean         0.990875
std          0.039343
min          0.500583
25%          0.996817
50%          0.999254
75%          0.999979
max          1.000000
Name: gmm_confidence, dtype: float64

Customers with GMM confidence below 80%:
218


#### Test Data Distribution Analysis

The test dataset contains 20,000 customer records with integrated supervised and unsupervised learning outputs.

The predicted churn probabilities exhibit substantial variation, with a mean of 46.11% and a standard deviation of 45.88%. This supports selecting test cases across different churn probability levels.

All three K-Means behavioral segments and all four GMM purchase personas are represented in the test dataset, enabling the evaluation of personalized retention recommendations across different customer profiles.

Additionally, 218 customers (1.09%) have a GMM membership confidence below 80%. Although these cases are relatively uncommon, their inclusion is important for evaluating how the prompt templates handle uncertain cluster assignments.

These observations support a targeted test case selection approach rather than random sampling, allowing the comparison to cover diverse customer profiles, churn probabilities, and levels of segmentation confidence.

In [19]:
high_risk = test_data["churn_probability"] >= 0.80
low_risk = test_data["churn_probability"] <= 0.20
intermediate_risk = test_data["churn_probability"].between(0.40, 0.60)

high_confidence = test_data["gmm_confidence"] >= 0.80
low_confidence = test_data["gmm_confidence"] < 0.80

persona = test_data["kmeans_persona"]

candidate_cases = {
    "TC01": test_data[
        high_risk & high_confidence &
        (persona == "Low-Engagement Customers")
    ],
    "TC02": test_data[
        high_risk & high_confidence &
        (persona == "High-Volume Buyers")
    ],
    "TC03": test_data[
        low_risk & high_confidence &
        (persona == "Engaged Light Buyers")
    ],
    "TC04": test_data[
        intermediate_risk & low_confidence
    ],
    "TC05": test_data[
        low_risk & high_confidence &
        (persona == "Low-Engagement Customers")
    ]
}

for case_id, candidates in candidate_cases.items():
    print(f"{case_id}: {len(candidates)} eligible customers")

TC01: 3099 eligible customers
TC02: 2826 eligible customers
TC03: 3312 eligible customers
TC04: 11 eligible customers
TC05: 2555 eligible customers


In [20]:
gmm_coverage = pd.DataFrame({
    case_id: candidates["gmm_persona"].value_counts()
    for case_id, candidates in candidate_cases.items()
}).fillna(0).astype(int)

display(gmm_coverage)

,TC01,TC02,TC03,TC04,TC05
gmm_persona,,,,,
Heavy Users,15,1744,56,5,13
Occasional Buyers,1091,0,1116,2,959
One-time Buyers,810,0,710,0,628
Regular Buyers,1183,1082,1430,4,955


In [21]:
selected_personas = {
    "TC01": "One-time Buyers",
    "TC02": "Heavy Users",
    "TC03": "Regular Buyers",
    "TC04": "Occasional Buyers",
    "TC05": "One-time Buyers"
}

selected_customers = []

for case_id, gmm_persona in selected_personas.items():

    eligible = candidate_cases[case_id][
        candidate_cases[case_id]["gmm_persona"] == gmm_persona
    ]

    customer = eligible.sample(
        n=1, random_state=42
    ).copy()

    customer.insert(0, "test_case", case_id)
    selected_customers.append(customer)

selected_test_cases = pd.concat(
    selected_customers, ignore_index=True
)

display(selected_test_cases)

,test_case,customer_id,recency_days,loyalty_score,cart_abandonment_rate,churn_prediction,churn_probability,kmeans_persona,gmm_persona,gmm_confidence
0,TC01,C056262,346,18,0.73,1,1.000000e+00,Low-Engagement Customers,One-time Buyers,0.999424
1,TC02,C090145,314,78,0.62,1,9.981688e-01,High-Volume Buyers,Heavy Users,1.000000
2,TC03,C015208,21,72,0.29,0,1.208694e-07,Engaged Light Buyers,Regular Buyers,0.996963
3,TC04,C085378,219,89,0.53,0,4.973180e-01,Engaged Light Buyers,Occasional Buyers,0.595011
4,TC05,C058518,210,72,0.20,0,3.233598e-02,Low-Engagement Customers,One-time Buyers,0.999957


#### Test Case Selection Rationale

Five customer records were selected from the test dataset using
a criteria-based sampling approach. Selection criteria were
defined to cover different churn probability levels, K-Means
behavioral segments, GMM purchase personas, and membership
confidence levels.

For testing purposes, churn probabilities were categorized
into the following selection ranges:

- Low: 0%–20%.
- Intermediate: 40%–60%.
- High: 80%–100%.

These ranges were selected to provide contrasting risk
scenarios and examine cases near the classification threshold.
They are used for test case selection rather than as official
business risk categories.

A GMM membership confidence below 80% was considered low,
consistent with the unsupervised learning analysis.

One customer was randomly selected from the eligible records
for each predefined scenario using a fixed random seed of 42
to ensure reproducibility.

Both prompt templates are evaluated using the same five
customer records to support a fair comparison.

#### Individual Test Case Justification

**TC01 – High-Risk, Low-Engagement Customer**

This customer has a very high predicted churn probability,
low loyalty score, and high cart abandonment rate. The
customer belongs to the Low-Engagement Customers segment
and One-time Buyers persona.

This case evaluates the ability of each prompt to generate
re-engagement and customer recovery recommendations that
reflect the customer's individual behavior and segmentation.

**TC02 – High-Risk, High-Volume Customer**

Despite having a relatively high loyalty score, this customer
has a high predicted churn probability of 99.82% and belongs
to the High-Volume Buyers segment and Heavy Users persona.

This case evaluates whether the prompts recognize the
customer's purchasing profile and provide appropriate
retention strategies rather than generic high-risk
recommendations.

**TC03 – Low-Risk, Engaged Customer**

This customer has a very low predicted churn probability,
recent activity, and a relatively high loyalty score.
The customer belongs to the Engaged Light Buyers segment
and Regular Buyers persona.

This case evaluates whether the prompts recommend maintaining
engagement and strengthening loyalty rather than unnecessary
intensive recovery interventions.

**TC04 – Uncertain GMM Membership**

This customer has a churn probability of 49.73% and a GMM
membership confidence of 59.50%.

This case evaluates whether the prompts acknowledge
segmentation uncertainty, avoid overreliance on the GMM
persona, and prioritize individual customer information
when generating recommendations.

**TC05 – Individual Risk versus Segment Risk**

This customer belongs to the same K-Means segment and GMM
persona as TC01 but has a substantially lower predicted
churn probability of 3.23%.

This case evaluates whether the prompts distinguish
individual churn risk from general segment characteristics
and avoid assuming that all customers within the same
segment have the same level of risk.

#### Test Case Coverage and Limitations

The selected cases cover all three K-Means behavioral
segments, all four GMM purchase personas, contrasting
churn probability levels, and uncertain GMM membership.

The five cases are intended to provide scenario coverage
rather than statistical representation of the entire
customer population. Since individual customer attributes
also vary across cases, differences in generated
recommendations cannot be attributed solely to segmentation.

### Preparing Prompt Inputs

In [23]:
test_customers = {}

for _, row in selected_test_cases.iterrows():
    test_customers[row["test_case"]] = {
        "churn_prediction": int(row["churn_prediction"]),
        "churn_probability": f'{row["churn_probability"]:.6%}',
        "recency_days": int(row["recency_days"]),
        "loyalty_score": int(row["loyalty_score"]),
        "cart_abandonment_rate": f'{row["cart_abandonment_rate"]:.2%}',
        "kmeans_persona": row["kmeans_persona"],
        "gmm_persona": row["gmm_persona"],
        "gmm_confidence": f'{row["gmm_confidence"]:.6%}'
    }

for case_id, customer_data in test_customers.items():
    print(f"\n{case_id}")
    print(customer_data)


TC01
{'churn_prediction': 1, 'churn_probability': '99.999998%', 'recency_days': 346, 'loyalty_score': 18, 'cart_abandonment_rate': '73.00%', 'kmeans_persona': 'Low-Engagement Customers', 'gmm_persona': 'One-time Buyers', 'gmm_confidence': '99.942420%'}

TC02
{'churn_prediction': 1, 'churn_probability': '99.816878%', 'recency_days': 314, 'loyalty_score': 78, 'cart_abandonment_rate': '62.00%', 'kmeans_persona': 'High-Volume Buyers', 'gmm_persona': 'Heavy Users', 'gmm_confidence': '100.000000%'}

TC03
{'churn_prediction': 0, 'churn_probability': '0.000012%', 'recency_days': 21, 'loyalty_score': 72, 'cart_abandonment_rate': '29.00%', 'kmeans_persona': 'Engaged Light Buyers', 'gmm_persona': 'Regular Buyers', 'gmm_confidence': '99.696265%'}

TC04
{'churn_prediction': 0, 'churn_probability': '49.731799%', 'recency_days': 219, 'loyalty_score': 89, 'cart_abandonment_rate': '53.00%', 'kmeans_persona': 'Engaged Light Buyers', 'gmm_persona': 'Occasional Buyers', 'gmm_confidence': '59.501126%'}

T

### Executing Prompt Tests

Each selected customer record is submitted to both the
Zero-Shot and Few-Shot prompt templates using the same
generative AI model and identical customer inputs.

The generated reports are collected for qualitative
and quantitative comparative analysis.

In [24]:
test_results = {}

for case_id, customer_data in test_customers.items():
    print(f"Generating reports for {case_id}...")

    test_results[case_id] = generate_customer_reports(
        customer_data
    )

    print(f"{case_id} completed successfully.")

Generating reports for TC01...
TC01 completed successfully.
Generating reports for TC02...
TC02 completed successfully.
Generating reports for TC03...
TC03 completed successfully.
Generating reports for TC04...
TC04 completed successfully.
Generating reports for TC05...
TC05 completed successfully.


### Saving Test Results

The generated reports are saved in a JSON file to preserve
the outputs and ensure consistency throughout the qualitative
and quantitative evaluation.

In [25]:
import json

results_path = Path("Generative_AI/test_results.json")

with results_path.open("w", encoding="utf-8") as file:
    json.dump(test_results, file, indent=4, ensure_ascii=False)

print("Test results saved successfully.")

Test results saved successfully.


In [26]:
with results_path.open("r", encoding="utf-8") as file:
    saved_results = json.load(file)

print("Number of test cases:", len(saved_results))
print("Test cases:", list(saved_results.keys()))

Number of test cases: 5
Test cases: ['TC01', 'TC02', 'TC03', 'TC04', 'TC05']


### Qualitative Evaluation

The Zero-Shot and Few-Shot reports are evaluated to assess the quality of their generated content and their suitability for executive decision-making.

The comparison highlights the strengths and weaknesses of each template, supported by observations from the generated reports.

#### Evaluation Criteria

The generated reports are qualitatively evaluated using the following five criteria:

1. **Relevance:** Examines whether the report's risk assessment and retention recommendations are relevant to the customer's churn prediction and available information.

2. **Detail & Completeness:** Assesses whether the report provides sufficient information, includes the required recommendations, and explains the rationale behind each recommendation.

3. **Clarity & Readability:** Evaluates whether the report uses clear, understandable, and well-organized language suitable for non-technical business executives.

4. **Personalization:** Examines whether the recommendations
   are tailored to the customer's individual characteristics,
   K-Means behavioral segment, GMM purchase persona, and
   membership confidence rather than providing generic advice.

5. **Safety & Factual Accuracy:** Assesses whether the report accurately represents the provided customer information, avoids unsupported assumptions or fabricated details, and does not present predictions or recommendations as guaranteed outcomes.

Each test case is analyzed by comparing the Zero-Shot and Few-Shot reports against these criteria. The observations are then summarized to identify the overall strengths and limitations of each template.

In [27]:
print("TC01 - Customer Input")
print(test_customers["TC01"])

print("\nTC01 - Generated Reports")

for prompt_type, report in saved_results["TC01"].items():
    print(f"\n--- {prompt_type} ---\n")
    print(report)

TC01 - Customer Input
{'churn_prediction': 1, 'churn_probability': '99.999998%', 'recency_days': 346, 'loyalty_score': 18, 'cart_abandonment_rate': '73.00%', 'kmeans_persona': 'Low-Engagement Customers', 'gmm_persona': 'One-time Buyers', 'gmm_confidence': '99.942420%'}

TC01 - Generated Reports

--- zero_shot_report ---

Risk Summary:
The predictive model estimates a very high churn probability (99.999998%) for this customer, classifying them as a churn risk (Prediction: 1). Key individual features show high disengagement: 346 days of inactivity (Recency Days), a low loyalty score of 18, and a 73.00% cart abandonment rate. Contextual segmentation indicates the customer belongs to the "Low-Engagement Customers" K-Means behavioral segment and is classified as a "One-time Buyer" via GMM, with a high membership confidence of 99.94%. 

Retention Recommendations:
1. Re-engagement Campaign: Deploy a targeted win-back email campaign offering assistance or a tailored incentive to address the 73

#### TC01 – Qualitative Evaluation

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Provides relevant win-back and loyalty-building recommendations aligned with the customer's very high predicted churn risk. | Provides relevant re-engagement and checkout assistance recommendations aligned with the customer's predicted churn risk and behavioral indicators. |
| Detail & Completeness | Includes all required sections and exactly two recommendations, with more detailed implementation suggestions. | Includes all required sections and exactly two recommendations, with concise explanations supporting each action. |
| Clarity & Readability | Well-structured and understandable, although some business terminology may be less accessible to non-expert readers. | Uses clearer and more direct language, making the recommendations easier to understand. |
| Personalization | Incorporates individual customer features, the K-Means behavioral segment, the GMM purchase persona, and membership confidence. | Incorporates the individual and segmentation information, with clearer links between specific customer characteristics and each recommendation. |
| Safety & Factual Accuracy | Accurately reports the main input values and avoids guaranteed outcomes. However, it presents inferred low purchase frequency as a directly observed pattern. | Accurately reports the main input values and avoids guaranteed outcomes. However, describing the purchase tier as reliable may overstate the meaning of high GMM membership confidence. |

**Summary:**

Both templates generate relevant, complete, and personalized
retention reports for TC01. The Zero-Shot template provides
more detailed implementation suggestions, while the Few-Shot
template offers clearer language and more explicit connections
between customer characteristics and recommended actions.

Both reports demonstrate generally appropriate use of
segmentation information, although minor limitations remain
in the interpretation of inferred purchasing behavior and
membership confidence. Both also interpret recency as general
inactivity rather than specifically time since the last purchase.

Overall, the Few-Shot template demonstrates a slight advantage
in clarity and explicit personalization for this test case,
while the Zero-Shot template provides greater implementation
detail.

In [28]:
print("TC02 - Customer Input")
print(test_customers["TC02"])

print("\nTC02 - Generated Reports")

for prompt_type, report in saved_results["TC02"].items():
    print(f"\n--- {prompt_type} ---\n")
    print(report)

TC02 - Customer Input
{'churn_prediction': 1, 'churn_probability': '99.816878%', 'recency_days': 314, 'loyalty_score': 78, 'cart_abandonment_rate': '62.00%', 'kmeans_persona': 'High-Volume Buyers', 'gmm_persona': 'Heavy Users', 'gmm_confidence': '100.000000%'}

TC02 - Generated Reports

--- zero_shot_report ---

Risk Summary:
The predictive model estimates a high churn probability of 99.82% for this customer. Individual features indicate a high recency of 314 days since their last activity, a loyalty score of 78, and a cart abandonment rate of 62.00%. The customer is categorized under the K-Means behavioral segment as a "High-Volume Buyer" and the GMM purchase tier as a "Heavy User," with 100.00% GMM membership confidence supporting this tier classification.

Retention Recommendations:
1. Re-engagement Campaign targeting abandoned carts: Send a personalized communication focusing on the 62.00% cart abandonment rate to encourage the completion of pending purchases after 314 days of inac

#### TC02 – Qualitative Evaluation

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Provides relevant cart recovery and loyalty-based recommendations aligned with the customer's high predicted churn risk. | Provides relevant win-back and checkout assistance recommendations aligned with the customer's predicted churn risk and purchasing profile. |
| Detail & Completeness | Includes all required sections and exactly two recommendations, with explicit rationales for each action. However, the binary churn prediction is not explicitly stated. | Includes all required sections and exactly two recommendations with concise supporting explanations. However, the binary churn prediction is not explicitly stated. |
| Clarity & Readability | Well-organized and informative, although some repetition occurs between the individual recommendation rationales and the strategic rationale. | Uses concise and direct language, making the report more accessible to non-technical business executives. |
| Personalization | Effectively incorporates individual customer features and segmentation outputs, particularly through targeted rewards tailored to Heavy Users. | Incorporates individual customer features and segmentation outputs, although the proposed loyalty incentive is less specific. |
| Safety & Factual Accuracy | Accurately reports the main input values but assumes the existence of pending purchases, interprets recency as general inactivity, and incorrectly describes all metrics as model estimates. | Accurately reports the main input values and explicitly avoids causal assumptions and guaranteed outcomes. However, it overstates the certainty associated with GMM membership confidence. |

**Summary:**

Both templates generate relevant and personalized retention
recommendations for TC02. Compared with TC01, both reports
demonstrate adaptation to the customer's higher loyalty score
and different purchasing profile despite similarly high
predicted churn probabilities.

The Zero-Shot template provides more detailed rationales and
more specific loyalty-based recommendations. However, it
introduces unsupported assumptions regarding pending purchases
and inaccurately describes all metrics as model estimates.

The Few-Shot template provides clearer and more concise
recommendations while explicitly acknowledging the limitations
of the suggested interventions. Nevertheless, its interpretation
of GMM membership confidence slightly overstates the certainty
of the segmentation information.

Overall, the Zero-Shot template demonstrates an advantage in
detail and recommendation specificity, while the Few-Shot
template demonstrates an advantage in clarity and factual
caution for this test case.

In [29]:
print("TC03 - Customer Input")
print(test_customers["TC03"])

print("\nTC03 - Generated Reports")

for prompt_type, report in saved_results["TC03"].items():
    print(f"\n--- {prompt_type} ---\n")
    print(report)

TC03 - Customer Input
{'churn_prediction': 0, 'churn_probability': '0.000012%', 'recency_days': 21, 'loyalty_score': 72, 'cart_abandonment_rate': '29.00%', 'kmeans_persona': 'Engaged Light Buyers', 'gmm_persona': 'Regular Buyers', 'gmm_confidence': '99.696265%'}

TC03 - Generated Reports

--- zero_shot_report ---

Risk Summary:
The model estimates a negligible churn probability of 0.000012% with a churn prediction of 0. The customer is relatively active with a recency of 21 days, a strong loyalty score of 72, and a cart abandonment rate of 29.00%. They are classified within the "Engaged Light Buyers" K-Means behavioral segment and assigned to the "Regular Buyers" GMM purchase tier with high confidence (99.696265%).

Retention Recommendations:
1. Implement targeted cart recovery reminders addressing the 29.00% cart abandonment rate to encourage completion of light purchases.
2. Introduce personalized product recommendations based on their "Regular Buyers" tier and loyalty score of 72 to

#### TC03 – Qualitative Evaluation

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Provides appropriate engagement maintenance and cart recovery recommendations aligned with the customer's very low predicted churn risk. | Provides appropriate loyalty reinforcement and checkout assistance recommendations, explicitly prioritizing ongoing engagement over aggressive churn recovery. |
| Detail & Completeness | Includes all eight input values, all required sections, and exactly two recommendations. | Includes all eight input values, all required sections, and exactly two recommendations. |
| Clarity & Readability | Presents a clear and structured report with understandable recommendations. | Uses concise and direct language and more explicitly communicates the purpose of the retention strategy. |
| Personalization | Directly incorporates the GMM purchase persona and loyalty score into personalized product recommendations, demonstrating stronger use of segmentation in the proposed actions. | Tailors recommendations to the loyalty score and cart abandonment rate, although segmentation information is more explicitly reflected in the strategic rationale than in the recommendations. |
| Safety & Factual Accuracy | Accurately reports the input values and avoids direct causal assumptions. However, the reference to completing light purchases is not directly supported by the individual customer inputs. | Accurately reports the input values and avoids guaranteed outcomes. However, it assumes existing reward benefits and may overstate the reliability of segmentation context based on membership confidence. |

**Summary:**

Both templates generate relevant and complete retention
reports that appropriately reflect the customer's very low
predicted churn risk.

Unlike the high-risk cases, both templates focus on
maintaining customer engagement rather than intensive
customer recovery.

The Zero-Shot template demonstrates stronger integration
of segmentation information into its personalized
recommendations, while the Few-Shot template provides
clearer language and more explicitly distinguishes
engagement maintenance from aggressive churn recovery.

Both templates accurately report the numerical inputs,
although minor unsupported assumptions remain in their
recommendations.

Overall, the Zero-Shot template demonstrates an advantage
in segmentation-based personalization, while the Few-Shot
template demonstrates an advantage in clarity and explicit
alignment with the customer's low-risk status.

In [30]:
print("TC04 - Customer Input")
print(test_customers["TC04"])

print("\nTC04 - Generated Reports")

for prompt_type, report in saved_results["TC04"].items():
    print(f"\n--- {prompt_type} ---\n")
    print(report)

TC04 - Customer Input
{'churn_prediction': 0, 'churn_probability': '49.731799%', 'recency_days': 219, 'loyalty_score': 89, 'cart_abandonment_rate': '53.00%', 'kmeans_persona': 'Engaged Light Buyers', 'gmm_persona': 'Occasional Buyers', 'gmm_confidence': '59.501126%'}

TC04 - Generated Reports

--- zero_shot_report ---

Risk Summary:
The model predicts no churn (Prediction: 0) with a churn probability estimate of 49.73%, indicating a moderate level of uncertainty. The customer has a high loyalty score of 89 and was last active 219 days ago, with a cart abandonment rate of 53.00%. They are categorized as an "Engaged Light Buyer" (K-Means) and an "Occasional Buyer" (GMM purchase tier). However, the GMM membership confidence is low at 59.50%, meaning greater reliance is placed on the individual customer features.

Retention Recommendations:
1. Re-engagement Campaign Targeting Cart Abandonment: Send a targeted email featuring items left in the cart to address the 53.00% cart abandonment rat

#### TC04 – Qualitative Evaluation

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Provides appropriate re-engagement and loyalty-based recommendations that address the customer's moderate churn probability and individual behavioral indicators. | Provides appropriate re-engagement and cart recovery recommendations that reflect the customer's borderline churn risk and individual characteristics. |
| Detail & Completeness | Includes all eight input values, all required sections, and exactly two recommendations. Explicitly acknowledges low GMM membership confidence. | Includes all eight input values, all required sections, and exactly two recommendations. Explicitly explains the implications of low GMM membership confidence. |
| Clarity & Readability | Presents a clear and structured report but communicates the borderline churn risk less explicitly. | Uses clear and direct language and more explicitly communicates the borderline nature of the predicted churn risk. |
| Personalization | Prioritizes individual customer features but still incorporates the uncertain Occasional Buyers persona into the second recommendation. | Bases both recommendations primarily on individual customer features and uses segmentation information as supporting context, demonstrating more consistent handling of low membership confidence. |
| Safety & Factual Accuracy | Accurately reports the numerical inputs but assumes the existence of items currently left in the cart and partially relies on an uncertain purchasing pattern. | Accurately reports the numerical inputs, avoids direct reliance on the uncertain GMM persona, and explicitly states that the interventions do not guarantee retention. However, it interprets recency as general inactivity. |

**Summary:**

Both templates generate relevant and complete retention
reports for a customer with borderline predicted churn
risk and low GMM membership confidence.

Both acknowledge the uncertainty associated with the
GMM purchase tier and recognize the importance of
prioritizing individual customer features.

However, the Zero-Shot template still incorporates the
uncertain purchasing persona into one recommendation,
creating a minor inconsistency between its stated
reasoning and proposed actions.

The Few-Shot template demonstrates more consistent
handling of segmentation uncertainty by grounding its
recommendations primarily in individual customer
features and using segmentation as supporting context.

Both reports interpret recency as general inactivity
rather than specifically time since the last purchase.

Overall, the Few-Shot template demonstrates an advantage
in uncertainty handling, clarity, and factual caution
for this test case.

In [31]:
print("TC05 - Customer Input")
print(test_customers["TC05"])

print("\nTC05 - Generated Reports")

for prompt_type, report in saved_results["TC05"].items():
    print(f"\n--- {prompt_type} ---\n")
    print(report)

TC05 - Customer Input
{'churn_prediction': 0, 'churn_probability': '3.233598%', 'recency_days': 210, 'loyalty_score': 72, 'cart_abandonment_rate': '20.00%', 'kmeans_persona': 'Low-Engagement Customers', 'gmm_persona': 'One-time Buyers', 'gmm_confidence': '99.995708%'}

TC05 - Generated Reports

--- zero_shot_report ---

Risk Summary:
The model predicts this customer will not churn (Churn Prediction: 0), with a low estimated churn probability of 3.23%. Relevant characteristics include a high loyalty score of 72, a moderate cart abandonment rate of 20.00%, and a recency of 210 days since the last activity. The customer is categorized within the "Low-Engagement Customers" K-Means behavioral segment and is classified with very high certainty (99.9957%) as a "One-time Buyer" under the GMM purchase tier.

Retention Recommendations:
1. Re-engagement email campaign with personalized product recommendations: Provide targeted messaging to address the 210-day gap in activity and the 20.00% cart a

#### TC05 – Qualitative Evaluation

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Provides appropriate re-engagement and second-purchase recommendations aligned with the customer's low predicted churn risk and purchasing profile. | Provides appropriate recommendations focused on encouraging repeat purchases and maintaining customer engagement without overstating churn risk. |
| Detail & Completeness | Includes all eight input values, all required sections, and exactly two recommendations, with more specific implementation suggestions. | Includes all eight input values, all required sections, and exactly two recommendations, although the proposed actions are less detailed. |
| Clarity & Readability | Presents a structured and understandable report, although some terminology, such as "low recency activity," is unclear. | Uses concise, direct, and accessible language suitable for non-technical business executives. |
| Personalization | Incorporates recency, loyalty score, cart abandonment rate, and segmentation information into the recommended actions, demonstrating more comprehensive use of customer characteristics. | Incorporates recency, loyalty score, and segmentation information, but does not explicitly use the cart abandonment rate to inform the recommendations or strategic rationale. |
| Safety & Factual Accuracy | Accurately reports the numerical inputs and avoids direct causal assumptions. However, it interprets recency as general inactivity and uses wording that may overstate the certainty of churn and segmentation predictions. | Accurately reports the numerical inputs and explicitly states that the interventions do not guarantee retention. However, describing the purchase tier as reliable based on membership confidence may overstate segmentation reliability. |

**Summary:**

Both templates generate relevant and personalized retention
reports that appropriately reflect the customer's low
predicted churn risk.

Despite sharing the same K-Means behavioral segment and
GMM purchase persona as TC01, both templates distinguish
between the customers' predicted churn risk levels and
adapt their recommendations accordingly.

The Zero-Shot template demonstrates more comprehensive
use of individual customer characteristics and provides
more specific retention actions.

The Few-Shot template provides clearer language and
demonstrates greater caution when communicating churn
predictions and expected intervention outcomes.

However, both templates exhibit limitations in their
interpretation of high GMM membership confidence.

Overall, the Zero-Shot template demonstrates an advantage
in detail and personalization, while the Few-Shot template
demonstrates an advantage in clarity and factual caution
for this test case.

### Overall Qualitative Comparison

| Criterion | Zero-Shot | Few-Shot |
|---|---|---|
| Relevance | Relevant recommendations across all five cases. | Relevant recommendations with clearer alignment to churn risk levels. |
| Detail & Completeness | More detailed and specific recommendations. | Complete but more concise recommendations. |
| Clarity & Readability | Clear but occasionally repetitive or complex. | Clearer and more accessible to business executives. |
| Personalization | Stronger integration of customer features and segmentation in several cases. | More consistent personalization under segmentation uncertainty. |
| Safety & Factual Accuracy | More frequent unsupported assumptions. | Greater factual caution, with some overstatement of segmentation reliability. |

Both templates successfully adapt their recommendations to
different customer profiles and churn risk levels. Zero-Shot
demonstrates strengths in detail and personalization, while
Few-Shot demonstrates strengths in clarity, factual caution,
and uncertainty handling.

However, these findings are limited to the five selected
test cases and do not establish the independent impact of
clustering on report quality.

Although the qualitative evaluation highlights the strengths
and limitations of both templates, these findings alone are
insufficient to determine the better-performing template.
The final selection should consider both qualitative and
quantitative evaluation results to ensure a comprehensive
and evidence-based decision.

### Quantitative Evaluation

To objectively compare the Zero-Shot and Few-Shot prompt templates, quantitative metrics are applied to the generated reports. Both templates are evaluated using the same test cases to ensure a consistent and fair comparison.


#### Response Length
Response length is measured using the word count of each
generated report. This metric helps compare the level of
verbosity and conciseness between the two templates.

A longer response does not necessarily indicate higher
quality, as excessive detail may reduce readability
and executive effectiveness.

In [32]:
length_results = []

for case_id, reports in saved_results.items():
    length_results.append({
        "Test Case": case_id,
        "Zero-Shot": len(reports["zero_shot_report"].split()),
        "Few-Shot": len(reports["few_shot_report"].split())
    })

length_df = pd.DataFrame(length_results)

display(length_df)

print("\nAverage Response Length (words):")
print(length_df[["Zero-Shot", "Few-Shot"]].mean().round(2))

,Test Case,Zero-Shot,Few-Shot
0,TC01,195,182
1,TC02,218,174
2,TC03,167,167
3,TC04,204,186
4,TC05,192,159



Average Response Length (words):
Zero-Shot    195.2
Few-Shot     173.6
dtype: float64


**Results and Interpretation:**

The Zero-Shot template generated an average of 195.2 words,
compared to 173.6 words for the Few-Shot template.

Few-Shot responses were approximately 11.07% shorter on
average and were shorter in four out of five test cases,
with equal response lengths in TC03.

These results support the qualitative findings that
Zero-Shot provides more detailed reports, while Few-Shot
favors conciseness, which may benefit time-constrained
business executives.

However, response length alone does not determine report
quality or executive effectiveness.

#### Keyword Alignment

Keyword alignment measures the coverage of relevant
customer retention concepts within the generated
recommendations and strategic rationale.

The risk summary and section headings are excluded
to reduce the influence of repeated input terminology
and predefined report structures.

Related keywords are grouped into domain concepts
to account for variations in wording. Each concept
is counted once, regardless of keyword frequency.

The keyword groups were manually defined for this
exploratory evaluation and were not independently
validated. Some concepts may overlap, and the results
may be influenced by the selected terminology.

In [33]:
import re

keyword_groups = {
    "Customer Re-engagement": [
        "re-engagement", "win-back", "reactivate"
    ],
    "Loyalty Reinforcement": [
        "reward", "loyalty benefit", "loyalty program"
    ],
    "Purchase Encouragement": [
        "repeat purchase", "second purchase"
    ],
    "Cart Recovery": [
        "cart recovery", "checkout", "cart reminder"
    ],
    "Incentives": [
        "incentive", "discount", "promotion"
    ],
    "Personalization": [
        "personalized", "tailored", "targeted"
    ],
    "Engagement Maintenance": [
        "maintain engagement", "sustain engagement",
        "continued interaction"
    ]
}

def calculate_keyword_alignment(report):
    content = report.split("Retention Recommendations:", 1)[1]
    content = content.replace("Strategic Rationale:", "")
    content = content.lower()

    matched_concepts = [
        concept
        for concept, terms in keyword_groups.items()
        if any(term in content for term in terms)
    ]

    score = len(matched_concepts) / len(keyword_groups) * 100

    return score


keyword_results = []

for case_id, reports in saved_results.items():
    keyword_results.append({
        "Test Case": case_id,
        "Zero-Shot (%)": calculate_keyword_alignment(
            reports["zero_shot_report"]
        ),
        "Few-Shot (%)": calculate_keyword_alignment(
            reports["few_shot_report"]
        )
    })

keyword_df = pd.DataFrame(keyword_results)

display(keyword_df.round(2))

print("\nAverage Keyword Alignment (%):")
print(
    keyword_df[["Zero-Shot (%)", "Few-Shot (%)"]]
    .mean().round(2)
)

,Test Case,Zero-Shot (%),Few-Shot (%)
0,TC01,57.14,71.43
1,TC02,57.14,57.14
2,TC03,28.57,42.86
3,TC04,42.86,57.14
4,TC05,42.86,71.43



Average Keyword Alignment (%):
Zero-Shot (%)    45.71
Few-Shot (%)     60.00
dtype: float64


**Results and Interpretation:**

The Few-Shot template achieved an average keyword
alignment score of 60.00%, compared to 45.71% for
the Zero-Shot template, outperforming it in four
out of five test cases and matching it in TC02.

Despite producing shorter reports, Few-Shot demonstrated
broader coverage of the selected retention concepts.

However, lower keyword alignment does not necessarily
indicate weaker recommendations, as not all retention
strategies are appropriate for every customer profile.

The metric also depends on the predefined keyword groups
and may overlook alternative expressions. Therefore,
these results should be interpreted alongside the
qualitative findings rather than as a standalone
indicator of recommendation quality.


#### 3. Readability Score (Flesch-Kincaid)

Readability is evaluated using the Flesch-Kincaid Grade Level, which estimates the educational reading level required to understand a text based on sentence length and word complexity.

Lower scores generally indicate easier-to-read text. This metric is used to compare the accessibility of the Zero-Shot and Few-Shot reports for business executives, although it does not directly measure the accuracy or usefulness of their recommendations.

In [ ]:
%pip install textstat

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for textstat from https://files.pythonhosted.org/packages/ca/31/0eb4cc5bb021b4ceaaa602c59ba16ce99256b9dd30981bef3f3a53d8555f/textstat-0.7.13-py3-none-any.whl.metadata
  Obtaining dependency information for pyphen from https://files.pythonhosted.org/packages/ec/1d/23801cf008f71575f0a4800463f349afa5f04b27fe783178a45cdc4d5edf/pyphen-0.18.1-py3-none-any.whl.metadata
  Obtaining dependency information for nltk from https://files.pythonhosted.org/packages/a2/04/e7816813e409ca874ba2fcc35455db0e4130b79eb277e1f9edd270ca3c69/nltk-3.10.4-py3-none-any.whl.metadata
  Obtaining dependency information for setuptools from https://files.pythonhosted.org/packages/ef/55/6dc6d6bb953bdfc52519a0059592c52b6e3b8ad51cf8256dffceb55b0610/setuptools-84.2.1-py3-none-any.whl.metadata
  Obtaining dependency information for defusedxml from https://files.pythonhosted.org/packages/07/6c/aa3f2f849e01cb6a001cd


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import textstat

readability_results = []

for case_id, reports in saved_results.items():
    readability_results.append({
        "Test Case": case_id,
        "Zero-Shot": textstat.flesch_kincaid_grade(
            reports["zero_shot_report"]
        ),
        "Few-Shot": textstat.flesch_kincaid_grade(
            reports["few_shot_report"]
        )
    })

readability_df = pd.DataFrame(readability_results)

display(readability_df.round(2))

print("\nAverage Flesch-Kincaid Grade Level:")
print(
    readability_df[["Zero-Shot", "Few-Shot"]]
    .mean().round(2)
)

,Test Case,Zero-Shot,Few-Shot
0,TC01,14.86,13.43
1,TC02,14.82,12.21
2,TC03,13.39,13.11
3,TC04,12.45,13.48
4,TC05,13.90,13.17



Average Flesch-Kincaid Grade Level:
Zero-Shot    13.88
Few-Shot     13.08
dtype: float64


**Results and Interpretation:**

The Few-Shot template achieved an average Flesch-Kincaid
Grade Level of 13.08, compared to 13.88 for Zero-Shot,
indicating a modest improvement in readability.

Few-Shot achieved lower readability scores in four out
of five test cases, with TC04 being the exception.

These results generally support the qualitative findings
regarding the clearer language of Few-Shot reports.
However, both templates require approximately a
college-level reading ability, indicating room for
improvement in language simplicity.

The metric measures linguistic complexity rather than
actual comprehension or executive effectiveness.

#### 4. User Preference Simulation

User preference simulation is conducted to estimate which
generated report a business executive would prefer when
making customer retention decisions.

For each test case, Gemini acts as a simulated business
executive and compares two anonymized reports (Report A
and Report B) generated for the same customer. The evaluator
selects the report it would prefer to use for making a clear,
practical retention decision, with the option to select a tie.

Each comparison is performed twice with reversed report
order to reduce position bias. Inconsistent preferences
are recorded as unstable results.

The preferences are recorded across the five test cases,
and the preference percentage for each template is calculated.

This approach simulates executive decision-making rather
than collecting actual user feedback. Since Gemini is also
used to generate the reports, the simulation may be affected
by model bias and should not be interpreted as independent
human validation.

In [ ]:
def build_evaluation_prompt(customer_data, report_a, report_b):
    return f"""
You are a business executive in the retail industry.

Your task is to compare two customer retention reports
generated for the same customer.

Customer Information:
{json.dumps(customer_data, indent=2)}

Report A:
{report_a}

Report B:
{report_b}

Evaluate both reports based on:
- Relevance to the customer's churn risk
- Detail and completeness
- Clarity and readability
- Personalization
- Safety and factual accuracy
- Practicality of the recommendations

Select the report you would prefer to use when making
a customer retention decision.

Do not favor a report simply because it is longer,
shorter, or presented first.

If both reports provide comparable decision-making
value, select TIE.

Return only a valid JSON object in the following format:

{{
    "preference": "A, B, or TIE",
    "justification": "Brief explanation of your preference"
}}
"""

In [36]:
def evaluate_reports(customer_data, report_a, report_b):

    evaluation_prompt = build_evaluation_prompt(
        customer_data, report_a, report_b
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=evaluation_prompt,
        config={
            "response_mime_type": "application/json",
            "temperature": 0
        }
    )

    evaluation = json.loads(response.text)

    preference = evaluation["preference"].strip().upper()

    if preference not in ["A", "B", "TIE"]:
        raise ValueError("Invalid evaluation preference")

    return {
        "preference": preference,
        "justification": evaluation["justification"]
    }

In [37]:
preference_results = []

for case_id, reports in saved_results.items():

    customer_data = test_customers[case_id]

    zero_shot = reports["zero_shot_report"]
    few_shot = reports["few_shot_report"]

    print(f"Evaluating {case_id}...")

    # First evaluation: A = Zero-Shot, B = Few-Shot
    evaluation_1 = evaluate_reports(
        customer_data, zero_shot, few_shot
    )

    # Second evaluation: A = Few-Shot, B = Zero-Shot
    evaluation_2 = evaluate_reports(
        customer_data, few_shot, zero_shot
    )

    # Map anonymous preferences to template names
    mapping_1 = {
        "A": "Zero-Shot",
        "B": "Few-Shot",
        "TIE": "Tie"
    }

    mapping_2 = {
        "A": "Few-Shot",
        "B": "Zero-Shot",
        "TIE": "Tie"
    }

    preference_1 = mapping_1[evaluation_1["preference"]]
    preference_2 = mapping_2[evaluation_2["preference"]]

    # Check preference consistency
    final_preference = (
        preference_1
        if preference_1 == preference_2
        else "Unstable"
    )

    preference_results.append({
        "Test Case": case_id,
        "First Evaluation": preference_1,
        "Second Evaluation": preference_2,
        "Final Preference": final_preference,
        "First Justification": evaluation_1["justification"],
        "Second Justification": evaluation_2["justification"]
    })

    print(f"{case_id}: {final_preference}")

Evaluating TC01...
TC01: Zero-Shot
Evaluating TC02...
TC02: Zero-Shot
Evaluating TC03...
TC03: Unstable
Evaluating TC04...
TC04: Unstable
Evaluating TC05...
TC05: Zero-Shot


In [38]:
preference_df = pd.DataFrame(preference_results)

display(preference_df[[
    "Test Case",
    "First Evaluation",
    "Second Evaluation",
    "Final Preference"
]])

preference_counts = (
    preference_df["Final Preference"]
    .value_counts()
    .reindex(
        ["Zero-Shot", "Few-Shot", "Tie", "Unstable"],
        fill_value=0
    )
)

preference_summary = pd.DataFrame({
    "Count": preference_counts,
    "Percentage (%)": (
        preference_counts / len(preference_df) * 100
    )
})

display(preference_summary)

,Test Case,First Evaluation,Second Evaluation,Final Preference
0,TC01,Zero-Shot,Zero-Shot,Zero-Shot
1,TC02,Zero-Shot,Zero-Shot,Zero-Shot
2,TC03,Few-Shot,Zero-Shot,Unstable
3,TC04,Zero-Shot,Few-Shot,Unstable
4,TC05,Zero-Shot,Zero-Shot,Zero-Shot


,Count,Percentage (%)
Final Preference,,
Zero-Shot,3,60.0
Few-Shot,0,0.0
Tie,0,0.0
Unstable,2,40.0


In [39]:
for result in preference_results:

    print("=" * 70)
    print("Test Case:", result["Test Case"])

    print("\nFirst Evaluation:", result["First Evaluation"])
    print(result["First Justification"])

    print("\nSecond Evaluation:", result["Second Evaluation"])
    print(result["Second Justification"])

    print("\nFinal Preference:", result["Final Preference"])

Test Case: TC01

First Evaluation: Zero-Shot
Report A provides a slightly more structured and cohesive strategic rationale, explicitly connecting the K-Means 'Low-Engagement' and GMM 'One-time Buyer' segments to actionable retention steps while maintaining accurate factual alignment with the customer's data.

Second Evaluation: Zero-Shot
Report B offers superior readability, slightly better integration of the customer's specific metrics into actionable recommendations, and clearly distinguishes between behavioral segments and tactical next steps. The structuring of the retention recommendations in Report B is more practical for execution by a retail marketing team.

Final Preference: Zero-Shot
Test Case: TC02

First Evaluation: Zero-Shot
Report A provides a more structured, readable, and detailed analysis. It explicitly separates the rationale for each retention recommendation, effectively synthesizes the customer's high-volume and heavy-user persona with their loyalty score, and trans

In [40]:
for result in preference_results[2:]:

    print("\nTest Case:", result["Test Case"])

    print("\nFirst Evaluation:", result["First Evaluation"])
    print(result["First Justification"])

    print("\nSecond Evaluation:", result["Second Evaluation"])
    print(result["Second Justification"])


Test Case: TC03

First Evaluation: Few-Shot
Report B provides a slightly more refined and practical approach for a low-risk, engaged customer. Instead of pushing aggressive personalized product recommendations based solely on tier classifications like Report A, Report B sensibly suggests maintaining standard loyalty communications and reinforcing existing rewards, which aligns better with the customer's strong loyalty score and very low churn probability.

Second Evaluation: Zero-Shot
Report B provides slightly more actionable and personalized recommendations, such as introducing personalized product recommendations based on the customer's tier, whereas Report A relies heavily on maintaining standard communications.

Test Case: TC04

First Evaluation: Zero-Shot
Report A provides a more structured and actionable approach to the retention recommendations, directly tying the high loyalty score and cart abandonment rate to specific campaign concepts. It also integrates the GMM confidence 

**Results and Interpretation:**

Zero-Shot achieved a consistent preference rate of 60%,
while Few-Shot achieved 0%. The remaining 40% of test
cases produced unstable preferences.

The evaluator favored Zero-Shot in TC01, TC02, and TC05
primarily because of its detailed, actionable recommendations
and explicit connections to customer characteristics.

In TC03 and TC04, preferences changed when the report
order was reversed, suggesting possible position bias
or inconsistency in evaluation priorities.

Although Few-Shot performed better in response length,
keyword alignment, and readability, the simulated evaluator
placed greater emphasis on recommendation specificity
and practical decision-making value in the three
consistently evaluated cases.

These findings should be interpreted cautiously due to
evaluation instability, the limited number of test cases,
and potential bias from using the same model for report
generation and evaluation.

#### Saving Preference Simulation Results

The simulated preferences and their corresponding
justifications are saved to ensure traceability and
allow further analysis without repeating API calls.

In [41]:
preference_path = Path("Generative_AI/preference_results.json")

with preference_path.open("w", encoding="utf-8") as file:
    json.dump(
        preference_results,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Preference simulation results saved successfully.")

Preference simulation results saved successfully.


### Overall Quantitative Comparison

| Metric | Zero-Shot | Few-Shot |
|---|---:|---:|
| Average Response Length | 195.2 words | 173.6 words |
| Average Keyword Alignment | 45.71% | 60.00% |
| Average Flesch-Kincaid Grade Level | 13.88 | 13.08 |
| Consistent Simulated User Preference | 60% | 0% |

The Few-Shot template produces shorter reports with broader
keyword coverage and slightly better readability. In contrast,
the Zero-Shot template achieves a higher consistent simulated
user preference due to its detailed and actionable recommendations.

However, 40% of the simulated preferences were unstable,
limiting the reliability of this metric.

The results demonstrate a trade-off between conciseness,
readability, terminology coverage, and practical decision-making
value. Therefore, the final template selection should consider
both qualitative and quantitative findings rather than relying
on a single metric.

## Best Prompt Selection & Justification

Based on the qualitative and quantitative evaluation results,
the Few-Shot template is selected for integration into the
final system. The selection is based on its overall alignment
with the system's objectives rather than its performance
on a single evaluation metric.

### Qualitative Strengths

The Few-Shot template consistently generates relevant,
complete, and personalized retention reports. It demonstrates
greater clarity, conciseness, and factual caution when
communicating model predictions and recommended actions.

A particularly important strength is its handling of
segmentation uncertainty in TC04, where it prioritizes
individual customer features over the low-confidence
GMM purchase persona.

Although Zero-Shot provides more detailed and specific
recommendations in several cases, it introduces more
unsupported assumptions. For an executive decision-support
system, clear and cautious interpretation of customer
information is prioritized over additional detail.

### Quantitative Evidence

The quantitative results further support the selection:

- **Response Length:** Few-Shot produces shorter reports,
  averaging 173.6 words compared to 195.2 words for
  Zero-Shot, representing an average reduction of 11.07%.

- **Keyword Alignment:** Few-Shot achieves 60.00% compared
  to 45.71% for Zero-Shot, demonstrating broader coverage
  of the selected retention concepts despite shorter reports.

- **Readability:** Few-Shot achieves a lower average
  Flesch-Kincaid Grade Level of 13.08 compared to 13.88,
  indicating slightly better readability.

- **User Preference Simulation:** Zero-Shot achieves a
  consistent preference rate of 60%, while Few-Shot
  achieves 0%, with the remaining 40% of cases producing
  unstable preferences. Across the ten individual
  evaluations, Zero-Shot was preferred eight times and
  Few-Shot twice.

The simulated evaluator favored Zero-Shot primarily for
its detailed and actionable recommendations. However,
the instability of some preferences and the potential
bias associated with using the same model for generation
and evaluation limit the strength of this evidence.

Therefore, the simulated preference results are considered
alongside the other evaluation findings rather than used
as the sole selection criterion.

### Alignment with System Goals

The system aims to support retail executives in making
informed customer retention decisions by transforming
machine learning outputs into understandable, personalized,
and actionable strategic insights.

Few-Shot aligns with these objectives by:

- Presenting customer churn predictions in clear and
  concise language.
- Generating personalized retention recommendations
  based on individual customer characteristics.
- Using segmentation information as supporting context
  rather than a direct determinant of churn risk.
- Demonstrating more cautious handling of uncertain
  model outputs.
- Supporting efficient decision-making through
  executive-oriented reports.

Although Zero-Shot offers greater recommendation detail
and stronger simulated user preference, Few-Shot provides
a more suitable balance between clarity, personalization,
factual caution, and practical decision support.

Therefore, Few-Shot is selected as the preferred template
for the final system based on the available evaluation
evidence and the system's priorities. This selection does
not imply that Few-Shot is universally superior, and
further evaluation with additional customer profiles and
actual business executives would strengthen the decision.